<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-26 15:58:56
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.55 C
CAGR/XIRR %:     1.7%
Est FTT Value:   2.74 C
Future Upside:   1.18 C (76.12%)
=================== PERFORMANCE ==========================
Current PnL:             -39.87 L (-20.77%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    9.36 L (6.28%)
CY Booked PnL:           6.36 L (4.09%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               47.19 K (0.3%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             3.24 L
Total Loss:               -43.11 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
50,M&M,-0.63,8.88,7.90,17.48,25898.0,26750.0,327825.0,3762.88,-3.36,68.0,X-LC,6.02,25.0,1.03,2.11,26.76,X40N,ATH,AUTO
76,UNITDSPR,-1.70,9.34,10.45,20.77,26754.0,21878.0,256022.0,1644.00,-4.23,63.0,X-MC,3.11,64.0,0.82,1.65,23.07,X40N,NTT,BREWERIES
12,BOSCH-HCIL,0.94,-24.17,31.85,-0.01,30869.0,-30886.0,96919.0,2282.00,757.51,48.0,M-SC,18.04,231.0,-1.00,0.62,64.97,OX40N,BTT,AC
75,TTKPRESTIG,-0.77,-32.24,47.71,0.09,32581.0,-32487.0,68290.0,770.00,-31.49,41.0,M-SC,3.05,250.0,-1.00,0.44,22.91,OX40N,NTT,DURABLES
52,MASFIN,-1.03,-21.43,54.33,21.26,41823.0,-21000.0,76980.0,396.02,-39.43,44.0,H-SC,13.76,167.0,-0.50,0.50,4.88,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-0.37,7.60,32.25,42.29,98398.0,21538.0,305109.0,2782.73,25.71,53.0,X-LC,3.75,29.0,0.22,1.96,25.59,X40N,ATH,AUTO
47,JIOFIN,1.86,-20.18,70.48,36.09,225905.0,-81021.0,320524.0,387.00,-29.17,56.0,X-LC,7.07,34.0,-0.36,2.06,11.00,XY24,BTT,FINANCE
4,ASIANPAINT,-2.98,-7.25,34.58,24.83,114318.0,-25827.0,330590.0,3422.40,-18.29,55.0,X-LC,3.04,30.0,-0.23,2.13,21.21,X40,ATH,PAINTS
44,INFY,4.69,-14.42,66.69,42.66,228793.0,-57785.0,343070.0,1972.00,-37.72,63.0,X-LC,5.23,3.0,-0.25,2.21,22.82,X40,NTT,IT
23,DMART,-1.64,-11.79,52.60,34.61,180261.0,-45808.0,342701.0,5391.45,-27.68,57.0,X-LC,13.37,14.0,-0.25,2.21,12.73,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
33,HCLTECH,6.42,0.49,25.09,25.70,91012.0,1771.0,362744.0,1837.07,-6.12,67.0,X-LC,9.31,8.0,0.02,2.33,44.73,X40,ATH,IT
71,TCS,4.89,-20.48,59.39,26.75,230507.0,-99951.0,388124.0,4208.34,-36.49,71.0,X-LC,6.21,2.0,-0.43,2.50,34.62,X40,ATH,IT
44,INFY,4.69,-14.42,66.69,42.66,228793.0,-57785.0,343070.0,1972.00,-37.72,63.0,X-LC,5.23,3.0,-0.25,2.21,22.82,X40,NTT,IT
8,BAJAJHLDNG,4.44,6.84,18.93,27.06,67768.0,22903.0,357990.0,14191.91,-7.44,65.0,X-LC,4.18,28.0,0.34,2.30,38.94,X40,ATH,FINANCE
24,EASEMYTRIP,4.29,-65.12,317.72,45.70,195941.0,-115144.0,61671.0,26.40,NaN,60.0,NaN,16.07,204.0,-0.59,0.40,18.57,XY24,NTT,TRAVEL


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-3.82,-29.40,204.78,115.17,391453.0,-79603.0,191158.0,1903.93,-23.39,47.0,H-SC,27.18,147.0,-0.20,1.23,68.98,AR,ATH,IT
48,KANSAINER,-3.46,-34.76,73.93,13.47,130067.0,-93735.0,175932.0,340.00,-29.00,52.0,H-SC,5.15,162.0,-0.72,1.13,24.35,XY24,NTT,PAINTS
4,ASIANPAINT,-2.98,-7.25,34.58,24.83,114318.0,-25827.0,330590.0,3422.40,-18.29,55.0,X-LC,3.04,30.0,-0.23,2.13,21.21,X40,ATH,PAINTS
53,NESTLEIND,-2.81,10.79,16.64,29.22,57561.0,33680.0,345919.0,1540.00,-20.86,56.0,X-LC,6.45,20.0,0.59,2.23,13.70,XY25,ATH,FMCG
20,DABUR,-2.67,-18.18,75.63,43.70,153508.0,-45096.0,202972.0,735.00,-30.64,67.0,X-MC,9.60,70.0,-0.29,1.31,18.05,XY24,BTT,FMCG


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.44,45.98,41.41,106.44,89253.0,67892.0,215536.0,1629.94,-8.75,41.0,M-SC,20.26,220.0,0.76,1.39,63.31,OX40N,ATH,CABLES
65,SIS,-1.06,-17.64,47.24,21.26,43198.0,-19589.0,91443.0,528.00,75.81,36.0,H-SC,4.69,168.0,-0.45,0.59,35.96,OX40N,NTT,MISC
12,BOSCH-HCIL,0.94,-24.17,31.85,-0.01,30869.0,-30886.0,96919.0,2282.00,757.51,48.0,M-SC,18.04,231.0,-1.00,0.62,64.97,OX40N,BTT,AC
42,INDIGOPNTS,-0.60,-30.46,43.83,0.02,53205.0,-53169.0,121390.0,1408.00,-33.68,49.0,M-SC,6.48,222.0,-1.00,0.78,38.31,OX40N,NTT,PAINTS
16,CERA,1.61,-31.76,81.54,23.89,97880.0,-55864.0,120039.0,9475.00,NaN,66.0,NaN,10.52,153.0,-0.57,0.77,18.45,OX40N,NTT,CERAMICS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.44,45.98,41.41,106.44,89253.0,67892.0,215536.0,1629.94,-8.75,41.0,M-SC,20.26,220.0,0.76,1.39,63.31,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.44,45.98,41.41,106.44,89253.0,67892.0,215536.0,1629.94,-8.75,41.0,M-SC,20.26,220.0,0.76,1.39,63.31,OX40N,ATH,CABLES
66,STARHEALTH,-0.73,9.29,34.74,47.26,106739.0,26128.0,307251.0,761.00,-20.27,54.0,H-SC,3.42,173.0,0.24,1.98,32.85,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.44,45.98,41.41,106.44,89253.0,67892.0,215536.0,1629.94,-8.75,41.0,M-SC,20.26,220.0,0.76,1.39,63.31,OX40N,ATH,CABLES
83,WIPRO,3.96,-9.68,113.70,93.01,256986.0,-24219.0,226021.0,420.00,-22.45,62.0,M-LC,7.45,98.0,-0.09,1.45,25.35,XR,NTT,IT
69,TANLA,-3.82,-29.40,204.78,115.17,391453.0,-79603.0,191158.0,1903.93,-23.39,47.0,H-SC,27.18,147.0,-0.20,1.23,68.98,AR,ATH,IT
41,INDIAMART,0.93,-34.05,204.68,100.92,166474.0,-42002.0,81334.0,4675.60,-60.70,44.0,H-SC,16.59,140.0,-0.25,0.52,3.19,AR,ATH,MISC
3,ALKYLAMINE,0.25,-37.35,207.33,92.54,131142.0,-37710.0,63253.0,4520.85,-47.27,28.0,H-SC,7.76,142.0,-0.29,0.41,20.11,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,0.18,-19.51,64.26,32.21,138814.0,-52359.0,216020.0,50690.44,-38.91,34.0,X-SC,7.94,55.0,-0.38,1.39,3.65,X40,ATH,APPARELS
37,HONAUT,-0.62,-24.61,100.15,50.88,145593.0,-47463.0,145375.0,58192.80,-40.91,34.0,X-SC,9.88,90.0,-0.33,0.94,10.76,X40N,ATH,ELECTRICAL
6,AWL,-0.22,-43.01,193.09,67.02,392058.0,-153265.0,203044.0,485.00,-74.90,40.0,X-SC,10.82,81.0,-0.39,1.31,0.99,XY24,NTT,FMCG
73,TMPV,-0.04,-34.67,129.01,49.60,401889.0,-165354.0,311518.0,600.00,-531.72,41.0,X-MC,16.80,4.0,-0.41,2.01,3.74,XY24,NTT,AUTO
11,BERGEPAINT,-0.28,-15.29,47.94,25.32,104669.0,-39415.0,218334.0,680.00,-24.28,42.0,X-MC,2.74,68.0,-0.38,1.41,14.49,XY24,NTT,PAINTS


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,2.00,-8.42,25.97,15.37,42539.0,-15055.0,163800.0,34390.26,-26.51,60.0,X-MC,0.27,58.0,-0.35,1.05,10.96,X40,ATH,PHARMA
11,BERGEPAINT,-0.28,-15.29,47.94,25.32,104669.0,-39415.0,218334.0,680.00,-24.28,42.0,X-MC,2.74,68.0,-0.38,1.41,14.49,XY24,NTT,PAINTS
17,CIPLA,1.17,9.00,13.66,23.89,44764.0,27045.0,327700.0,1648.14,43.06,67.0,X-MC,2.95,18.0,0.60,2.11,22.74,X40N,ATH,PHARMA
4,ASIANPAINT,-2.98,-7.25,34.58,24.83,114318.0,-25827.0,330590.0,3422.40,-18.29,55.0,X-LC,3.04,30.0,-0.23,2.13,21.21,X40,ATH,PAINTS
76,UNITDSPR,-1.70,9.34,10.45,20.77,26754.0,21878.0,256022.0,1644.00,-4.23,63.0,X-MC,3.11,64.0,0.82,1.65,23.07,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.22,-43.01,193.09,67.02,392058.0,-153265.0,203044.0,485.00,-74.90,40.0,X-SC,10.82,81.0,-0.39,1.31,0.99,XY24,NTT,FMCG
54,PAGEIND,0.18,-19.51,64.26,32.21,138814.0,-52359.0,216020.0,50690.44,-38.91,34.0,X-SC,7.94,55.0,-0.38,1.39,3.65,X40,ATH,APPARELS
73,TMPV,-0.04,-34.67,129.01,49.60,401889.0,-165354.0,311518.0,600.00,-531.72,41.0,X-MC,16.80,4.0,-0.41,2.01,3.74,XY24,NTT,AUTO
26,GILLETTE,-0.03,-14.81,51.06,28.69,110435.0,-37611.0,216285.0,10891.01,-38.94,50.0,X-SC,6.18,83.0,-0.34,1.39,4.50,X40,ATH,FMCG
55,PGHH,1.92,-45.80,141.71,31.02,164009.0,-97784.0,115736.0,17484.41,-59.32,57.0,X-SC,21.03,57.0,-0.60,0.74,8.95,X40,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,0.03,-59.25,231.80,35.20,121535.0,-76239.0,52431.0,2096.00,-18.59,48.0,X-SC,6.33,91.0,-0.63,0.34,8.96,X40,NTT,FOOTWEAR
55,PGHH,1.92,-45.80,141.71,31.02,164009.0,-97784.0,115736.0,17484.41,-59.32,57.0,X-SC,21.03,57.0,-0.60,0.74,8.95,X40,ATH,FMCG
37,HONAUT,-0.62,-24.61,100.15,50.88,145593.0,-47463.0,145375.0,58192.80,-40.91,34.0,X-SC,9.88,90.0,-0.33,0.94,10.76,X40N,ATH,ELECTRICAL
2,ACC,0.10,-43.45,209.12,74.81,317093.0,-116496.0,151632.0,3906.00,21.06,61.0,X-SC,10.87,85.0,-0.37,0.98,12.33,XY24,BTT,CEMENT
1,ABBOTINDIA,2.00,-8.42,25.97,15.37,42539.0,-15055.0,163800.0,34390.26,-26.51,60.0,X-MC,0.27,58.0,-0.35,1.05,10.96,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,-0.73,-12.59,47.23,28.70,180665.0,-55073.0,382522.0,452.00,-17.14,74.0,X-LC,4.71,1.0,-0.30,2.46,20.39,X40,NTT,FMCG
71,TCS,4.89,-20.48,59.39,26.75,230507.0,-99951.0,388124.0,4208.34,-36.49,71.0,X-LC,6.21,2.0,-0.43,2.50,34.62,X40,ATH,IT
44,INFY,4.69,-14.42,66.69,42.66,228793.0,-57785.0,343070.0,1972.00,-37.72,63.0,X-LC,5.23,3.0,-0.25,2.21,22.82,X40,NTT,IT
73,TMPV,-0.04,-34.67,129.01,49.60,401889.0,-165354.0,311518.0,600.00,-531.72,41.0,X-MC,16.80,4.0,-0.41,2.01,3.74,XY24,NTT,AUTO
33,HCLTECH,6.42,0.49,25.09,25.70,91012.0,1771.0,362744.0,1837.07,-6.12,67.0,X-LC,9.31,8.0,0.02,2.33,44.73,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,0.94,-24.17,31.85,-0.01,30869.0,-30886.0,96919.0,2282.00,757.51,48.0,M-SC,18.04,231.0,-1.00,0.62,64.97,OX40N,BTT,AC
79,VALIANTORG,1.48,-30.76,151.50,74.15,171631.0,-50317.0,113288.0,838.00,NaN,38.0,NaN,33.46,145.0,-0.29,0.73,68.59,XR,NTT,CHEMICALS
69,TANLA,-3.82,-29.40,204.78,115.17,391453.0,-79603.0,191158.0,1903.93,-23.39,47.0,H-SC,27.18,147.0,-0.20,1.23,68.98,AR,ATH,IT
25,FINCABLES,-0.44,45.98,41.41,106.44,89253.0,67892.0,215536.0,1629.94,-8.75,41.0,M-SC,20.26,220.0,0.76,1.39,63.31,OX40N,ATH,CABLES
27,GLAXO,2.42,11.59,21.60,35.69,48035.0,23100.0,222384.0,3380.16,-18.90,62.0,X-MC,5.63,56.0,0.48,1.43,32.17,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
81,VOLTAS,-1.37,-4.91,52.69,45.20,96067.0,-9417.0,182325.0,1856.00,3.71,66.0,X-MC,14.68,79.0,-0.10,1.17,28.46,XY25,NTT,AC
34,HDFCBANK,1.87,8.21,24.48,34.70,95802.0,29696.0,391348.0,1073.00,-17.12,73.0,X-LC,5.23,26.0,0.31,2.52,26.73,X40,BTT,BANKS
51,MARUTI,-1.30,3.14,22.16,25.99,77643.0,10665.0,350375.0,17120.06,6.60,73.0,X-LC,7.95,33.0,0.14,2.26,25.47,XR,ATH,AUTO
61,REPCOHOME,3.15,-19.14,134.76,89.82,312181.0,-54848.0,231657.0,880.00,NaN,62.0,NaN,7.68,161.0,-0.18,1.49,25.05,XY24,NTT,FINANCE
50,M&M,-0.63,8.88,7.90,17.48,25898.0,26750.0,327825.0,3762.88,-3.36,68.0,X-LC,6.02,25.0,1.03,2.11,26.76,X40N,ATH,AUTO


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,23.54
1,25,52.43
2,50,82.86


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.90
SC    27.76
MC    21.69
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.87
XY24     22.53
X40N     19.31
XY25     10.26
XR        9.82
OX40N     6.50
AR        3.04
SR        0.68
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    38.45
X-MC    20.29
H-SC    13.69
X-SC     8.67
M-SC     5.40
M-LC     1.45
M-MC     1.05
L-MC     0.35
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.82,-20.46,63.91
IT,12.81,-28.86,96.00
FINANCE,10.92,-20.09,71.20
AUTO,9.21,-11.76,52.82
MISC,5.46,-45.36,103.79
INSURANCE,5.46,2.52,34.11
PAINTS,5.45,-25.07,47.53
BANKS,5.29,-9.05,63.73
PHARMA,4.59,4.92,18.96


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3996952.0,19
X40,1999710.0,16
XR,1581233.0,13
X40N,1433525.0,12
AR,1032919.0,6
OX40N,779552.0,10
XY25,695637.0,6
SR,308665.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
H-SC,2320667.0,15
X-LC,2276019.0,17
X-MC,1815948.0,14
X-SC,1456883.0,8
M-SC,531893.0,7
M-LC,256986.0,1
M-MC,81790.0,1
L-MC,47535.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,,sum,count
Conviction,Criteria,,
X-LC,X40,1138857.0,8
H-SC,XY24,993756.0,5
X-SC,XY24,709151.0,2
X-MC,XY24,660066.0,3
H-SC,AR,557927.0,2
X-MC,X40N,550947.0,5
X-SC,X40,534793.0,4
X-LC,X40N,498642.0,4
H-SC,XR,389619.0,4


In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 38.0 seconds
